### Example: one accumulator in a race model
#### 0.1 Simulate and visualize the model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cssm import race_multistage
from ssms.basic_simulators.race_math import big_F, q, small_f

RANDOM_STATE = 20260820
OMISSION = -999.0

In [ ]:
mu = 0.75
sigma = 1.0
x0 = 0.0
a = 1.0
b = 0.0
T = 1.5
dt = 1e-3
num_trajs = 2_000

def simulate_trajs(mu, sigma, x0, T, dt, num, rng):
    """Euler-Maruyama paths for the trajectory plot."""
    t_grid = np.arange(0.0, T + dt, dt)
    X = np.empty((num, t_grid.size))
    X[:, 0] = x0
    for step in range(1, t_grid.size):
        X[:, step] = X[:, step - 1] + mu * dt + sigma * np.sqrt(dt) * rng.normal(size=num)
    return t_grid, X

t_grid, X_grids = simulate_trajs(mu, sigma, x0, T, dt, num_trajs, np.random.default_rng(RANDOM_STATE))
expected_mean = x0 + mu * t_grid
empirical_mean = X_grids.mean(axis=0)
expected_std = sigma * np.sqrt(t_grid)
empirical_std = X_grids.std(axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_grid, empirical_mean, label='empirical mean', color='tab:blue')
ax.plot(t_grid, expected_mean, label='expected mean', linewidth=2.5, linestyle='--', color='tab:blue')
ax.plot(t_grid, empirical_mean + 2 * empirical_std, label=r'empirical mean $\pm$ 2 std', color='tab:green')
ax.plot(t_grid, empirical_mean - 2 * empirical_std, color='tab:green')
ax.plot(t_grid, expected_mean + 2 * expected_std, label=r'expected mean $\pm$ 2 std', linewidth=2.5, linestyle='--', color='tab:green')
ax.plot(t_grid, expected_mean - 2 * expected_std, linewidth=2.5, linestyle='--', color='tab:green')
ax.plot(t_grid, X_grids[:10, :].T, alpha=0.5)
ax.plot(t_grid, a + b * t_grid, color='black', linestyle='--', label='upper boundary')
ax.autoscale(axis='x', tight=True)
ax.set(xlabel='time', ylabel='evidence')
ax.legend(fontsize=9)
plt.show()

#### 0.2 Simulate first-passage times, compute FPTD, CDF, and NPD

In [ ]:
race_arrays = {
    'mu_array': np.array([[[mu]]]),
    'sigma_array': np.array([[[sigma]]]),
    'node_array': np.zeros((1, 1, 1)),
    'd_array': np.ones((1, 1), dtype=np.int32),
    'upper_intercept_array': np.array([[[a]]]),
    'upper_slope_array': np.array([[[b]]]),
    'x0_array': np.array([[x0]]),
}

num_fpt = 50_000
out = race_multistage(
    **race_arrays, n_samples=num_fpt, delta_t=dt, max_t=T, random_state=RANDOM_STATE,
)
rt = out['rts'].reshape(-1)
x_final = out['metadata']['x_final'].reshape(-1)
exited = rt != OMISSION
fp_times = rt[exited]
np_positions = x_final[~exited]

print(f'empirical F(T): {exited.mean():.4f}')
print(f'analytic F(T): {float(big_F(T, mu, sigma, a, b, T, x0)):.4f}')

In [ ]:
ts = np.linspace(1e-3, T, 1_000)
xs = np.linspace(-4.0, a + b * T, 1_000)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fpt_bins = np.linspace(0.0, T, 101)
fpt_width = fpt_bins[1] - fpt_bins[0]
fpt_counts, _ = np.histogram(fp_times, bins=fpt_bins)
ax[0].stairs(fpt_counts / (num_fpt * fpt_width), fpt_bins, color='black', label='Monte Carlo')
ax[0].plot(ts, small_f(ts, mu, sigma, a, b, T, x0), color='tab:blue', label=r'$f(t)$')
ax[0].set(xlabel=r'$t$', ylabel='FPTD')
ax[0].legend()

np_bins = np.linspace(-4.0, a + b * T, 101)
np_width = np_bins[1] - np_bins[0]
np_counts, _ = np.histogram(np_positions, bins=np_bins)
ax[1].stairs(np_counts / (num_fpt * np_width), np_bins, color='gray', label='Monte Carlo')
ax[1].plot(xs, q(xs, mu, sigma, a, b, T, x0), color='tab:red', label=r'$q(x, T)$')
ax[1].set(xlabel=r'$x$', ylabel='NPD')
ax[1].legend()
plt.show()

In [ ]:
empirical_cdf = np.array([(fp_times <= t).sum() / num_fpt for t in ts])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ts, empirical_cdf, color='black', label='Monte Carlo')
ax.plot(ts, big_F(ts, mu, sigma, a, b, T, x0), color='tab:blue', label=r'$F(t)$')
ax.set(xlabel=r'$t$', ylabel='CDF', ylim=(0, 1))
ax.legend()
plt.show()